In [1]:
from strategies import *
from scipy.signal import savgol_filter
import pandas as pd

if __name__ == '__main__':
    
    df = pd.read_csv("../../backtesting/data/GOOGL/GOOGL.USUSD_Candlestick_5_M_ASK_05.10.2022-05.10.2024.csv")

    df['Gmt time']=df["Gmt time"].str.replace(".000","")
    df['Gmt time']=pd.to_datetime(df['Gmt time'],format='%d.%m.%Y %H:%M:%S')
    datetime_est=pd.to_datetime(df["Gmt time"], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern')
    
    fromTodayStart = '2022-10-05 09:30:00'
    toNow   = '2022-12-05 16:00:00'
    df = df[datetime_est.between(fromTodayStart, toNow)].copy()
        
    df.reset_index(drop=True, inplace=True)
    df['index']=df.index
    
    df.rename(columns={"Open": "open"}, inplace=True)
    df.rename(columns={"High": "high"}, inplace=True)
    df.rename(columns={"Low": "low"}, inplace=True)
    df.rename(columns={"Volume": "volume"}, inplace=True)
    df.rename(columns={"Close": "close"}, inplace=True)
    df.rename(columns={"Gmt time": "time"}, inplace=True)
    df.set_index("time", inplace=True, drop=True)
    
    df = df[df.notnull().all(axis=1)]
    df=df[(df.volume != 0)]
    df=df[df.high!=df.low]
    
    df = df[['index','high','low','close','volume','open']] 

    result = squeez(df)

    log_sequence = []
    
    for candle in range(0, len(result)):
        
        position = (result.iloc[candle]).position
        date = (result.iloc[candle]).date_est
        time = (result.iloc[candle]).time_est
        buying_price = (result.iloc[candle]).buying_price
        selling_price = (result.iloc[candle]).selling_price
        
        if (position==1):
            trans = [date,time,candle,'GOOGLE', 10, round(buying_price, 2), 'B']
        elif (position==-1):  
            trans = [date, time, candle,'GOOGLE', 10, round(selling_price, 2), 'S']
        else:
            trans = [date, time, candle,'GOOGLE', 10, 0, 'N']
            
        log_sequence.append(trans)

    df_log_sequence = pd.DataFrame(log_sequence, columns=['Date','Time','Candle','Symbol','Quantity','Price', 'Side'])
    df_log_sequence.to_csv('live_test_script.csv', index=False)
    
    #print(log_sequence)   